# 03 用户级时序特征工程

## 本 notebook 目标

本 notebook 的目标是将原始行为日志转换为用户级建模样本。

样本粒度：

> 一行 = 一个用户在某个 prediction_date 的预测样本。

建模逻辑：

- 观察窗口：prediction_date 前 7 天；
- 标签窗口：prediction_date 当天；
- 特征：用户在观察窗口内的行为特征；
- 标签：用户在 prediction_date 当天是否购买。

这样设计可以模拟真实业务中的次日购买预测场景：

> 用过去 7 天行为，预测用户明天是否购买。

关键原则：

> 特征只能使用 prediction_date 之前的数据，避免未来信息泄漏。

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../outputs/tables", exist_ok=True)

clean_data_path = "../data/processed/ecommerce_behavior_2019_10_01_15_clean.csv"

df = pd.read_csv(clean_data_path)

In [13]:
# 1. 时间处理
df["event_time"] = pd.to_datetime(df["event_time"], utc=True).dt.tz_convert(None)
df["date"] = df["event_time"].dt.floor("D")
df["hour"] = df["event_time"].dt.hour

## 样本构造逻辑

对于每一个 `prediction_date`，本文构造如下样本：

- 观察窗口：`prediction_date` 前 7 天；
- 标签窗口：`prediction_date` 当天；
- 特征：用户在观察窗口内的行为统计、活跃度、商品/品牌/类目丰富度、价格偏好和最近行为间隔；
- 标签：用户在预测日当天是否发生购买行为。

该设计保证了特征发生在标签之前，从而避免未来信息泄漏。

In [14]:
# 2. 构造单日用户级样本函数
def build_user_sample_for_one_day(df, prediction_date, window_days=7):
    """
    构造某一个 prediction_date 的用户级训练样本。

    样本粒度：
        一行 = 一个用户在某个 prediction_date 的预测样本。

    观察窗口：
        [prediction_date - window_days, prediction_date - 1]

    标签窗口：
        prediction_date 当天。

    标签定义：
        label = 1：用户在 prediction_date 当天发生过 purchase；
        label = 0：用户在 prediction_date 当天没有发生 purchase。

    注意：
        特征只使用 prediction_date 之前的数据，避免未来信息泄漏。
    """

    prediction_date = pd.to_datetime(prediction_date).floor("D")

    obs_start_date = prediction_date - pd.Timedelta(days=window_days)
    obs_end_date = prediction_date - pd.Timedelta(days=1)

    # --------------------------------------------------------
    # 1. 观察窗口和标签窗口
    # --------------------------------------------------------

    obs_data = df[
        (df["date"] >= obs_start_date) &
        (df["date"] <= obs_end_date)
    ].copy()

    label_data = df[
        df["date"] == prediction_date
    ].copy()

    # 防止数据泄漏：观察窗口不能包含 prediction_date 当天或未来日期
    if len(obs_data) > 0:
        assert obs_data["date"].max() < prediction_date, \
            "观察窗口包含 prediction_date 当天或未来日期，存在数据泄漏风险"

    # --------------------------------------------------------
    # 2. 行为次数特征
    # --------------------------------------------------------

    event_cnt_features = (
        obs_data
        .groupby(["user_id", "event_type"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    event_cnt_features = event_cnt_features.rename(columns={
        "view": "user_view_cnt_7d",
        "cart": "user_cart_cnt_7d",
        "purchase": "user_purchase_cnt_7d"
    })

    for col in [
        "user_view_cnt_7d",
        "user_cart_cnt_7d",
        "user_purchase_cnt_7d"
    ]:
        if col not in event_cnt_features.columns:
            event_cnt_features[col] = 0

    # --------------------------------------------------------
    # 3. 用户活跃度与兴趣广度特征
    # --------------------------------------------------------

    unique_features = (
        obs_data
        .groupby("user_id")
        .agg(
            user_unique_item_cnt_7d=("item_id", "nunique"),
            user_unique_category_cnt_7d=("category_id", "nunique"),
            user_unique_category_code_cnt_7d=("category_code", "nunique"),
            user_unique_brand_cnt_7d=("brand", "nunique"),
            user_active_days_7d=("date", "nunique"),
            user_active_hours_7d=("hour", "nunique"),
            user_session_cnt_7d=("user_session", "nunique")
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 4. 价格偏好特征
    # --------------------------------------------------------

    price_features = (
        obs_data
        .groupby("user_id")
        .agg(
            user_price_mean_7d=("price", "mean"),
            user_price_median_7d=("price", "median"),
            user_price_max_7d=("price", "max"),
            user_price_min_7d=("price", "min"),
            user_price_std_7d=("price", "std")
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 5. 不同行为下的商品、品牌、类目丰富度
    # --------------------------------------------------------

    behavior_features = (
        obs_data
        .groupby(["user_id", "event_type"])
        .agg(
            item_nunique=("item_id", "nunique"),
            brand_nunique=("brand", "nunique"),
            category_nunique=("category_id", "nunique")
        )
        .reset_index()
    )

    behavior_wide = behavior_features.pivot(
        index="user_id",
        columns="event_type",
        values=["item_nunique", "brand_nunique", "category_nunique"]
    )

    behavior_wide.columns = [
        f"user_{event}_{metric}_7d"
        for metric, event in behavior_wide.columns
    ]

    behavior_wide = behavior_wide.reset_index()

    # --------------------------------------------------------
    # 6. 最近一次总体行为间隔
    # --------------------------------------------------------

    last_event = (
        obs_data
        .groupby("user_id")["event_time"]
        .max()
        .reset_index(name="last_event_time")
    )

    prediction_datetime = pd.to_datetime(prediction_date)

    last_event["user_last_event_gap_hours"] = (
        prediction_datetime - last_event["last_event_time"]
    ).dt.total_seconds() / 3600

    last_event = last_event[["user_id", "user_last_event_gap_hours"]]

    # --------------------------------------------------------
    # 7. 最近一次 view/cart/purchase 行为间隔
    # --------------------------------------------------------

    last_event_by_type = (
        obs_data
        .groupby(["user_id", "event_type"])["event_time"]
        .max()
        .reset_index()
    )

    last_event_by_type["gap_hours"] = (
        prediction_datetime - last_event_by_type["event_time"]
    ).dt.total_seconds() / 3600

    last_gap_wide = last_event_by_type.pivot(
        index="user_id",
        columns="event_type",
        values="gap_hours"
    )

    last_gap_wide = last_gap_wide.rename(columns={
        "view": "user_last_view_gap_hours",
        "cart": "user_last_cart_gap_hours",
        "purchase": "user_last_purchase_gap_hours"
    }).reset_index()

    for col in [
        "user_last_view_gap_hours",
        "user_last_cart_gap_hours",
        "user_last_purchase_gap_hours"
    ]:
        if col not in last_gap_wide.columns:
            last_gap_wide[col] = np.nan

    # --------------------------------------------------------
    # 8. 合并特征
    # --------------------------------------------------------

    features = event_cnt_features.copy()

    merge_tables = [
        unique_features,
        price_features,
        behavior_wide,
        last_event,
        last_gap_wide
    ]

    for temp_table in merge_tables:
        features = features.merge(
            temp_table,
            on="user_id",
            how="left"
        )

    # --------------------------------------------------------
    # 9. 行为率特征
    # --------------------------------------------------------

    features["user_cart_rate_7d"] = np.where(
        features["user_view_cnt_7d"] > 0,
        features["user_cart_cnt_7d"] / features["user_view_cnt_7d"],
        0
    )

    features["user_purchase_rate_7d"] = np.where(
        features["user_view_cnt_7d"] > 0,
        features["user_purchase_cnt_7d"] / features["user_view_cnt_7d"],
        0
    )

    features["user_purchase_per_cart_7d"] = np.where(
        features["user_cart_cnt_7d"] > 0,
        features["user_purchase_cnt_7d"] / features["user_cart_cnt_7d"],
        0
    )

    features["user_view_per_active_day_7d"] = np.where(
        features["user_active_days_7d"] > 0,
        features["user_view_cnt_7d"] / features["user_active_days_7d"],
        0
    )

    # --------------------------------------------------------
    # 10. 构造标签
    # --------------------------------------------------------

    label = (
        label_data[label_data["event_type"] == "purchase"]
        .groupby("user_id")
        .size()
        .reset_index(name="purchase_cnt_label")
    )

    label["label"] = 1

    sample = features.merge(
        label[["user_id", "label"]],
        on="user_id",
        how="left"
    )

    sample["label"] = sample["label"].fillna(0).astype(int)
    sample["prediction_date"] = prediction_date

    # --------------------------------------------------------
    # 11. 缺失值处理
    # --------------------------------------------------------
    # gap 缺失表示用户过去 7 天没有发生该类行为。
    # 用较大的时间间隔表示“很久没有发生”。

    gap_cols = [
        "user_last_event_gap_hours",
        "user_last_view_gap_hours",
        "user_last_cart_gap_hours",
        "user_last_purchase_gap_hours"
    ]

    for col in gap_cols:
        if col in sample.columns:
            sample[col] = sample[col].fillna(window_days * 24 + 24)

    num_cols = sample.select_dtypes(include=[np.number]).columns
    sample[num_cols] = sample[num_cols].fillna(0)

    return sample

## 特征体系说明

本项目构造的用户级特征主要包括以下几类：

1. 行为频次特征：刻画用户过去 7 天浏览、加购、购买的次数；
2. 活跃度特征：刻画用户活跃天数、活跃小时数和 session 数；
3. 商品兴趣特征：刻画用户浏览或购买过的商品、品牌和类目丰富度；
4. 价格偏好特征：刻画用户近期交互商品的价格水平；
5. 最近行为间隔特征：刻画用户距离最近一次浏览、加购或购买的时间；
6. 行为转化率特征：刻画用户从浏览到加购、购买的历史转化倾向。

这些特征共同描述了用户近期活跃程度、兴趣强度和历史购买倾向，是后续购买预测模型的核心输入。

In [21]:
# 3. 构造多个 prediction_date 的样本
prediction_dates = pd.date_range(
    start="2019-10-08",
    end="2019-10-15",
    freq="D"
)

sample_list = []

for pred_date in prediction_dates:

    one_day_sample = build_user_sample_for_one_day(
        df=df,
        prediction_date=pred_date,
        window_days=7
    )

    sample_list.append(one_day_sample)

model_data = pd.concat(sample_list, axis=0, ignore_index=True)

In [25]:
# 4. 样本检查
print("建模样本维度:", model_data.shape)

print("\n标签数量分布:")
print(model_data["label"].value_counts())

print("\n标签比例分布:")
print(model_data["label"].value_counts(normalize=True))

print("\nprediction_date 分布:")
print(model_data["prediction_date"].value_counts().sort_index())

print("\n缺失值数量 Top 20:")
print(model_data.isnull().sum().sort_values(ascending=False).head(20))

display(model_data.head())

建模样本维度: (8178887, 35)

标签数量分布:
label
0    8090566
1      88321
Name: count, dtype: int64

标签比例分布:
label
0    0.989201
1    0.010799
Name: proportion, dtype: float64

prediction_date 分布:
prediction_date
2019-10-08     957543
2019-10-09     981575
2019-10-10    1002441
2019-10-11    1026060
2019-10-12    1042100
2019-10-13    1047455
2019-10-14    1057754
2019-10-15    1063959
Name: count, dtype: int64

缺失值数量 Top 20:
user_id                             0
user_cart_cnt_7d                    0
user_purchase_cnt_7d                0
user_view_cnt_7d                    0
user_unique_item_cnt_7d             0
user_unique_category_cnt_7d         0
user_unique_category_code_cnt_7d    0
user_unique_brand_cnt_7d            0
user_active_days_7d                 0
user_active_hours_7d                0
user_session_cnt_7d                 0
user_price_mean_7d                  0
user_price_median_7d                0
user_price_max_7d                   0
user_price_min_7d                   0
user_price_

,user_id,user_cart_cnt_7d,user_purchase_cnt_7d,user_view_cnt_7d,user_unique_item_cnt_7d,user_unique_category_cnt_7d,user_unique_category_code_cnt_7d,user_unique_brand_cnt_7d,user_active_days_7d,user_active_hours_7d,user_session_cnt_7d,user_price_mean_7d,user_price_median_7d,user_price_max_7d,user_price_min_7d,user_price_std_7d,user_cart_item_nunique_7d,user_purchase_item_nunique_7d,user_view_item_nunique_7d,user_cart_brand_nunique_7d,user_purchase_brand_nunique_7d,user_view_brand_nunique_7d,user_cart_category_nunique_7d,user_purchase_category_nunique_7d,user_view_category_nunique_7d,user_last_event_gap_hours,user_last_cart_gap_hours,user_last_purchase_gap_hours,user_last_view_gap_hours,user_cart_rate_7d,user_purchase_rate_7d,user_purchase_per_cart_7d,user_view_per_active_day_7d,label,prediction_date
0,183503497,0,0,1,1,1,0,0,1,1,1,15.770,15.770,15.77,15.77,0.000000,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,122.283333,192.0,192.0,122.283333,0.0,0.0,0.0,1.0,0,2019-10-08
1,184265397,0,0,4,2,1,1,1,1,1,1,127.675,127.675,143.89,111.46,18.723469,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,1.0,78.148889,192.0,192.0,78.148889,0.0,0.0,0.0,4.0,0,2019-10-08
2,208669541,0,0,2,2,2,1,1,1,1,1,13.610,13.610,18.47,8.75,6.873078,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,2.0,90.170556,192.0,192.0,90.170556,0.0,0.0,0.0,2.0,0,2019-10-08
3,216064734,0,0,4,2,2,1,2,1,1,1,662.775,662.775,818.48,507.07,179.792647,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0.0,2.0,4.783056,192.0,192.0,4.783056,0.0,0.0,0.0,4.0,0,2019-10-08
4,222907508,0,0,2,2,1,1,1,1,1,1,18.470,18.470,18.47,18.47,0.000000,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,1.0,17.560278,192.0,192.0,17.560278,0.0,0.0,0.0,2.0,0,2019-10-08


In [23]:
# 5. 保存建模样本和特征列表
model_data_path = "../data/processed/model_data_user_level_7d.csv"

model_data.to_csv(
    model_data_path,
    index=False,
    encoding="utf-8-sig"
)

exclude_cols = ["user_id", "prediction_date", "label"]

feature_cols = [
    col for col in model_data.columns
    if col not in exclude_cols
]

feature_info = pd.DataFrame({
    "feature_name": feature_cols
})

feature_info.to_csv(
    "../outputs/tables/user_level_feature_list.csv",
    index=False,
    encoding="utf-8-sig"
)

print("特征数量:", len(feature_cols))
display(feature_info)

特征数量: 32


,feature_name
0,user_cart_cnt_7d
1,user_purchase_cnt_7d
2,user_view_cnt_7d
3,user_unique_item_cnt_7d
4,user_unique_category_cnt_7d
5,user_unique_category_code_cnt_7d
6,user_unique_brand_cnt_7d
7,user_active_days_7d
8,user_active_hours_7d
9,user_session_cnt_7d


# 本节小结

1. 本 notebook 将行为日志转换为用户级监督学习样本。
2. 每一行样本代表一个用户在某个 prediction_date 的预测样本。
3. 特征来自 prediction_date 前 7 天，标签来自 prediction_date 当天。
4. 该设计模拟真实业务中的次日购买预测，并避免未来信息泄漏。
5. 构造的特征覆盖行为频次、活跃度、兴趣广度、价格偏好、最近行为间隔和历史转化率。